# Week 8 Concepts: LangChain Framework Basics

This notebook is the runnable companion to the four daily notes. Every code cell was actually executed in this environment -- `langchain-core` 0.3.86 installed via `pip install langchain-core`, running on Python 3.9.6 -- except one clearly marked cell in the Day 2 section, which needs a real API key and is shown only as a schematic. The outputs shown below are the real, captured results of running each cell, not transcribed or predicted values.

In [ ]:
# Suppress a noisy (and harmless) urllib3/LibreSSL warning so it doesn't
# clutter every cell's output below.
import warnings
warnings.filterwarnings("ignore")

import langchain_core
print("langchain_core version:", langchain_core.__version__)


langchain_core version: 0.3.86


## Day 1: LangChain and the LCEL Pipe

`Runnable.__or__` composes two Runnables into a `RunnableSequence`: calling `.invoke()` on the sequence runs the first step, then feeds its output into the second step's input. Below: the mechanism reproduced from scratch in a tiny class (`MiniRunnable`) with nothing hidden, then the same composition pattern run for real against installed `langchain_core`.

In [ ]:
class MiniRunnable:
    """Local stand-in for langchain_core.runnables.Runnable: keeps only the
    one mechanism this lesson is about -- __or__ building a sequence that
    still exposes .invoke()."""

    def invoke(self, value):
        raise NotImplementedError

    def __or__(self, other):
        # a | b -> MiniSequence(a, b): nothing more magical than this.
        return MiniSequence(self, other)


class MiniSequence(MiniRunnable):
    def __init__(self, first, second):
        self.first = first
        self.second = second

    def invoke(self, value):
        # The whole trick behind `prompt | llm | parser`: the first step's
        # return value becomes the second step's argument.
        first_output = self.first.invoke(value)   # step 1 runs fully
        return self.second.invoke(first_output)     # its output feeds step 2


class FuncStep(MiniRunnable):
    """Wraps a plain function as a pipeline step -- real LangChain does this
    coercion automatically (as a RunnableLambda) when you pipe into a
    callable, so you rarely have to write this wrapper by hand."""

    def __init__(self, fn):
        self.fn = fn

    def invoke(self, value):
        return self.fn(value)


class TinyTemplate(MiniRunnable):
    """Local stand-in for PromptTemplate.from_template(...)."""

    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        # dict -> str : fills {placeholders} the same way str.format does
        return self.template.format(**variables)


class StubForecaster(MiniRunnable):
    """Mock 'model': matches the .invoke() shape a real chat model would
    have, but returns a scripted string -- zero network calls, fully
    deterministic, good enough to prove the wiring works."""

    def invoke(self, prompt_text):
        return f"[stub-forecast] based on '{prompt_text[:40]}...', expect steady demand"


weather_prompt = TinyTemplate("Given these signals: {signals}, forecast next week's sales.")
trim_and_shout = FuncStep(lambda text: text.strip().upper())

# dict -> str (filled template) -> str (forecast) -> str (upper-cased)
forecast_chain = weather_prompt | StubForecaster() | trim_and_shout
print(type(forecast_chain).__name__)  # -> MiniSequence, itself a MiniRunnable

print(forecast_chain.invoke({"signals": "rising foot traffic, no promotions running"}))
print(forecast_chain.invoke({"signals": "holiday week, heavy discounting"}))


MiniSequence
[STUB-FORECAST] BASED ON 'GIVEN THESE SIGNALS: RISING FOOT TRAFFIC...', EXPECT STEADY DEMAND
[STUB-FORECAST] BASED ON 'GIVEN THESE SIGNALS: HOLIDAY WEEK, HEAVY...', EXPECT STEADY DEMAND


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable

class ScriptedModel(Runnable):
    """Stands in for a chat model: same .invoke() shape as a real model,
    zero network calls, fully deterministic output for testing the wiring
    before spending a single real API call."""
    def invoke(self, prompt_value, config=None, **kwargs):
        # prompt_value: StringPromptValue (PromptTemplate's output type)
        text = prompt_value.to_string()  # StringPromptValue -> str
        return f"MOCK-REPLY: read {len(text)} chars starting '{text[:24]}'"

def to_upper(text: str) -> str:
    return text.upper()

prompt = PromptTemplate.from_template("Explain {topic} in one short sentence.")
pv = prompt.invoke({"topic": "binary search"})
print("prompt.invoke ->", type(pv).__name__, ":", pv)

# RunnableLambda coercion happens automatically here: to_upper is a plain
# function, not a hand-built Runnable, but piping into it still works.
chain = prompt | ScriptedModel() | to_upper
print("chain type:", type(chain).__name__)  # -> RunnableSequence

# dict -> StringPromptValue -> str -> str
result = chain.invoke({"topic": "binary search"})
print("result:", result)


prompt.invoke -> StringPromptValue : text='Explain binary search in one short sentence.'
chain type: RunnableSequence
result: MOCK-REPLY: READ 44 CHARS STARTING 'EXPLAIN BINARY SEARCH IN'


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

class FakeChatModel(Runnable):
    """Matches a real chat model's .invoke() contract: consumes a
    ChatPromptValue (or list[BaseMessage]) and returns an AIMessage."""
    def invoke(self, input_value, config=None, **kwargs):
        # ChatPromptValue -> list[BaseMessage], len=2 (system, human)
        messages = input_value.to_messages()
        last_human = next(
            (m.content for m in reversed(messages) if isinstance(m, HumanMessage)),
            "",
        )
        return AIMessage(content=f"[fake-llm] you said: {last_human}")

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a terse assistant."),
    ("human", "{question}"),
])

# Inspect each intermediate shape explicitly, one .invoke() at a time:
pv = chat_prompt.invoke({"question": "What is LCEL?"})
print("ChatPromptValue.to_messages():", pv.to_messages())

msg = FakeChatModel().invoke(pv)
print("model output:", type(msg).__name__, "content=", msg.content)

parsed = StrOutputParser().invoke(msg)
print("parser output:", type(parsed).__name__, ":", parsed)

# Now the same three steps as one composed chain:
# dict -> ChatPromptValue -> AIMessage -> str
chain = chat_prompt | FakeChatModel() | StrOutputParser()
result = chain.invoke({"question": "What is LCEL?"})
print("chain result:", type(result).__name__, ":", result)


ChatPromptValue.to_messages(): [SystemMessage(content='You are a terse assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is LCEL?', additional_kwargs={}, response_metadata={})]
model output: AIMessage content= [fake-llm] you said: What is LCEL?
parser output: str : [fake-llm] you said: What is LCEL?
chain result: str : [fake-llm] you said: What is LCEL?


In [ ]:
import time
from langchain_core.runnables import RunnableLambda

def slow_double(x):
    time.sleep(0.2)  # stands in for a slow network call to a model API
    return x * 2

r = RunnableLambda(slow_double)
t0 = time.time()
out = r.batch([1, 2, 3, 4])   # runs the 4 calls concurrently via a thread pool
elapsed = time.time() - t0
print("results:", out)
print(f"elapsed: {elapsed:.2f}s (4 sequential calls would take roughly 0.8s)")


results: [2, 4, 6, 8]
elapsed: 0.21s (4 sequential calls would take roughly 0.8s)


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RunnableParallel runs several Runnables against the *same* input and
# collects their outputs into a dict; RunnablePassthrough just returns its
# input unchanged, which is how you carry the original value forward
# alongside a transformed one (the backbone of many retrieval chains).
branch = RunnableParallel(
    upper=RunnableLambda(lambda x: x.upper()),
    length=RunnableLambda(lambda x: len(x)),
    original=RunnablePassthrough(),
)
print(branch.invoke("hello world"))


{'upper': 'HELLO WORLD', 'length': 11, 'original': 'hello world'}


## Day 2: Reusable Templates, Swappable Models, Manual Parsing

A template is defined once and reused with different inputs via `.batch()`. Swapping model providers is mostly a construction-line change because every chat model class implements the same `Runnable` contract (shown schematically below -- it needs real installs and API keys to actually run). Then: parsing and validating raw model output by hand, and a direct, run-for-real comparison against `JsonOutputParser` to see exactly what a framework parser does and does not check.

In [ ]:
review_prompt = PromptTemplate.from_template(
    "Rate the sentiment of this review from 1-5 and give one reason: {review}"
)
reviews = [
    {"review": "The battery died after two days."},
    {"review": "Fast shipping, exactly as described."},
]
# .batch() runs the template over a list of inputs -- idiomatic LCEL,
# not a Python for-loop. Same template object, different {review} value.
for pv in review_prompt.batch(reviews):
    print(pv.to_string())

# .partial() bakes in a fixed variable and returns a *new* PromptTemplate;
# `base` itself is left untouched.
base = PromptTemplate.from_template("You are a {persona}. Answer: {question}")
support_bot = base.partial(persona="support agent")
print(support_bot.invoke({"question": "how do I reset my password?"}).to_string())
print("base is untouched:", base.input_variables)  # -> still needs 'persona' too


Rate the sentiment of this review from 1-5 and give one reason: The battery died after two days.
Rate the sentiment of this review from 1-5 and give one reason: Fast shipping, exactly as described.
You are a support agent. Answer: how do I reset my password?
base is untouched: ['persona', 'question']


The cell below cannot be executed in this sandbox -- it needs `langchain-openai` and `langchain-anthropic` installed plus real API keys for both providers. It's included as real, syntactically correct code you would actually run; every line is commented out so the notebook still executes top-to-bottom with no network call and no key required.

In [ ]:
# NOT EXECUTED IN THIS SANDBOX -- needs real installs + API keys for both.
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
#
# model_a = ChatOpenAI(model="gpt-4o-mini")
# model_b = ChatAnthropic(model="claude-3-5-haiku-20241022")
# for model in (model_a, model_b):
#     response = model.invoke("Summarize LCEL in one sentence.")
#     print(response.content)  # both return AIMessage -- same .content read
print("skipped: requires langchain-openai / langchain-anthropic + real API keys")


skipped: requires langchain-openai / langchain-anthropic + real API keys


In [ ]:
import json

def parse_and_validate(raw: str) -> dict:
    """Hand-rolled parsing + validation -- no framework output parser."""
    data = json.loads(raw)  # raises json.JSONDecodeError on malformed JSON

    required = {"label", "confidence", "tags"}
    missing = required - data.keys()
    if missing:
        raise ValueError(f"missing keys: {missing}")

    if data["label"] not in {"positive", "neutral", "negative"}:
        raise ValueError(f"unexpected label: {data['label']}")

    conf = data["confidence"]
    if not isinstance(conf, (int, float)) or isinstance(conf, bool) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence out of range: {conf}")

    if not isinstance(data["tags"], list):
        raise ValueError("tags must be a list")

    return data  # -> dict, all four invariants checked, safe to use downstream

good = '{"label": "positive", "confidence": 0.82, "tags": ["shipping", "praise"]}'
print(parse_and_validate(good))

# "amazing" is well-formed JSON but not one of the three labels this
# system understands -- parse_and_validate correctly rejects it.
bad = '{"label": "amazing", "confidence": 0.82, "tags": ["shipping"]}'
try:
    parse_and_validate(bad)
except ValueError as e:
    print("correctly rejected:", e)


{'label': 'positive', 'confidence': 0.82, 'tags': ['shipping', 'praise']}
correctly rejected: unexpected label: amazing


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

# Truncated object (missing closing brace) -- json.loads would reject this
# outright, but JsonOutputParser is built for streaming partial JSON and
# recovers from it.
print("recovered from truncation:",
      parser.invoke('{"priority": "high", "category": "billing", "eta_minutes": 15'))

# Syntactically valid, semantically wrong ("urgent" isn't a real priority,
# eta_minutes is a string not an int) -- passes straight through, no error.
print("semantically bad, no error:",
      parser.invoke('{"priority": "urgent", "eta_minutes": "soon"}'))

# Genuinely non-JSON text -- this is the one case that does raise.
try:
    parser.invoke("Sure! Here is the answer: not json at all")
except Exception as e:
    print("correctly raised:", type(e).__name__)


recovered from truncation: {'priority': 'high', 'category': 'billing', 'eta_minutes': 15}
semantically bad, no error: {'priority': 'urgent', 'eta_minutes': 'soon'}
correctly raised: OutputParserException


## Day 3: A Safe Calculator Tool and an FAQ Tool

`eval()` is unsafe for user-supplied expressions. `ast.literal_eval` is *also* not enough for a calculator: it only parses literal constants and containers, not an arithmetic operation between two literals -- confirmed for real below, `ast.literal_eval("12 * 8")` genuinely raises `ValueError`. The fix is a custom AST walker that whitelists arithmetic nodes and rejects everything else (names, calls, attribute access, subscripts, ...), also verified below against real attack strings.

In [ ]:
import ast

print(ast.literal_eval("12"))          # -> 12          (a literal: fine)
print(ast.literal_eval("[1, 2, 3]"))   # -> [1, 2, 3]    (a literal container: fine)

try:
    ast.literal_eval("12 * 8")  # BinOp (multiplication) isn't part of
                                 # literal_eval's restricted grammar at all
except ValueError as e:
    print("literal_eval('12 * 8') raises:", e)


12
[1, 2, 3]
literal_eval('12 * 8') raises: malformed node or string: <ast.BinOp object at 0x10a374490>


In [ ]:
import operator

# Whitelist: exactly the arithmetic this calculator supports, nothing else.
_BIN_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _eval(node):
    # ast.parse(..., mode="eval") always wraps the real expression in an
    # Expression node -- unwrap it once and recurse into .body.
    if isinstance(node, ast.Expression):
        return _eval(node.body)

    # Explicitly exclude bool: isinstance(True, int) is True in Python, so
    # without this check "True * 8" would silently evaluate to 8.
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
            and not isinstance(node.value, bool):
        return node.value  # -> int | float

    if isinstance(node, ast.BinOp) and type(node.op) in _BIN_OPS:
        left = _eval(node.left)    # recurse: left side may itself be a BinOp
        right = _eval(node.right)  # recurse: right side may itself be a BinOp
        return _BIN_OPS[type(node.op)](left, right)

    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval(node.operand))

    # Names, Call, Attribute, Subscript, List, Compare, ... all fall
    # through to here and are rejected.
    raise ValueError(f"disallowed expression: {type(node).__name__}")

def calc(expr: str):
    tree = ast.parse(expr, mode="eval")  # str -> ast.Expression
    return _eval(tree)                    # ast.Expression -> int | float

for expr in ["12 * 8", "(9 - 3) ** 2 / 4", "-8 + 20 / 4", "100 % 7", "2 ** 10"]:
    print(f"{expr:20} => {calc(expr)}")

print()
# Real attack strings -- every one is rejected by name, not by pattern-
# matching "looks dangerous": Attribute/Call/List/Name were simply never
# added to the whitelist above.
for bad in ["__import__('os').system('echo pwned')", "(1).__class__.__bases__",
            "open('secrets.txt')", "[1, 2, 3]", "a + 1"]:
    try:
        calc(bad)
        print("SHOULD NOT REACH:", bad)
    except ValueError as e:
        print(f"rejected: {bad!r:45} -> {e}")

print()
try:
    calc("1/0")
except ZeroDivisionError as e:
    # calc() correctly evaluates the Div node; the failure is the division
    # itself at runtime, not a disallowed-expression rejection -- a real
    # caller needs to catch this alongside ValueError.
    print("division by zero surfaces as:", type(e).__name__, "->", e)


12 * 8               => 96
(9 - 3) ** 2 / 4     => 9.0
-8 + 20 / 4          => -3.0
100 % 7              => 2
2 ** 10              => 1024

rejected: "__import__('os').system('echo pwned')"       -> disallowed expression: Call
rejected: '(1).__class__.__bases__'                     -> disallowed expression: Attribute
rejected: "open('secrets.txt')"                         -> disallowed expression: Call
rejected: '[1, 2, 3]'                                   -> disallowed expression: List
rejected: 'a + 1'                                       -> disallowed expression: Name

division by zero surfaces as: ZeroDivisionError -> division by zero


In [ ]:
_FAQ = [
    (("refund", "money back"), "Refunds post within 5-7 business days after we receive the return."),
    (("hours", "open"), "Support is staffed 9am-6pm, Monday through Friday."),
    (("shipping", "delivery"), "Standard shipping takes 3-5 business days."),
]

def faq(question: str):
    q = question.lower()  # case-insensitive keyword match
    for keywords, answer in _FAQ:
        if any(kw in q for kw in keywords):
            return answer  # -> str: first matching answer, in list order
    return None  # -> None: no keyword matched anything

def handle(user_input: str) -> str:
    has_digit = any(c.isdigit() for c in user_input)
    has_operator = any(op in user_input for op in "+-*/")
    if has_digit and has_operator:
        try:
            return f"= {calc(user_input)}"
        except (ValueError, ZeroDivisionError) as e:
            return f"couldn't evaluate that: {e}"
    answer = faq(user_input)
    return answer if answer else "no matching tool for that yet."

# The last input is the router's known blind spot: no digits, no operator
# symbols, and no FAQ keyword -- even though a human reads it instantly as
# an arithmetic question.
for text in ["9 * 6", "can I renew my loan?", "what are your hours",
             "what is nine times six"]:
    print(f"{text!r:30} -> {handle(text)}")


'9 * 6'                        -> = 54
'can I renew my loan?'         -> no matching tool for that yet.
'what are your hours'          -> Support is staffed 9am-6pm, Monday through Friday.
'what is nine times six'       -> no matching tool for that yet.


## Day 4: Memory, Growth, and Persistence

Each model call is stateless -- "memory" means re-sending prior turns as part of the next prompt, so an unbounded conversation makes every later prompt bigger (and slower, and costlier), measured for real below. Two bounding strategies: keep a recent window, or periodically fold older turns into a summary. Persisting history (e.g. to SQLite) lets a conversation survive a restart -- but anything a user typed, PII included, is stored verbatim unless you actively redact it, which matters for data-retention and compliance, and a real (imperfect) regex redaction pass makes that limitation concrete.

In [ ]:
def build_prompt(history, new_message):
    convo = "\n".join(f"{role}: {text}" for role, text in history)
    return f"{convo}\nuser: {new_message}\nassistant:"

history = []
sizes = []
for turn in range(1, 21):
    user_msg = f"question {turn} about the order"
    prompt = build_prompt(history, user_msg)  # re-serializes everything so far
    sizes.append(len(prompt))
    history.append(("user", user_msg))
    history.append(("assistant", f"answer {turn}"))

print("turn 1 prompt length:", sizes[0], "chars")
print("turn 20 prompt length:", sizes[-1], "chars")
print("growth factor:", round(sizes[-1] / sizes[0], 1), "x over 20 turns")
print("full history length:", len(history), "entries")


turn 1 prompt length: 44 chars
turn 20 prompt length: 1071 chars
growth factor: 24.3 x over 20 turns
full history length: 40 entries


In [ ]:
def windowed_history(history, max_turns=6):
    """Mitigation 1: keep only the most recent N turns, drop the rest."""
    return history[-max_turns:]  # -> list, len <= max_turns

def compact_history(history, keep_recent=6, summarize=None):
    """Mitigation 2: collapse older turns into one running summary line."""
    if len(history) <= keep_recent:
        return history  # nothing to compact yet
    old, recent = history[:-keep_recent], history[-keep_recent:]
    old_text = "\n".join(f"{r}: {t}" for r, t in old)
    # `summarize` would typically be another LLM call in a real system --
    # defaults to crude truncation here so this cell needs no extra model.
    summary = summarize(old_text) if summarize else old_text[:200] + "..."
    return [("system", f"earlier conversation summary: {summary}")] + list(recent)

print("full history:", len(history), "entries")
print("windowed turns kept:", len(windowed_history(history)))
print("compacted turns kept:", len(compact_history(history)))


full history: 40 entries
windowed turns kept: 6
compacted turns kept: 7


In [ ]:
import sqlite3
import os

def save_history_sqlite(history, db_path, conversation_id="demo"):
    """Persist history so it survives a process restart."""
    conn = sqlite3.connect(db_path)
    conn.execute(
        "CREATE TABLE IF NOT EXISTS turns "
        "(conversation_id TEXT, turn_index INTEGER, speaker TEXT, text TEXT)"
    )
    conn.execute("DELETE FROM turns WHERE conversation_id = ?", (conversation_id,))
    conn.executemany(
        "INSERT INTO turns VALUES (?, ?, ?, ?)",
        [(conversation_id, i, speaker, text) for i, (speaker, text) in enumerate(history)],
    )
    conn.commit()
    conn.close()

def load_history_sqlite(db_path, conversation_id="demo"):
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT speaker, text FROM turns WHERE conversation_id = ? ORDER BY turn_index",
        (conversation_id,),
    ).fetchall()
    conn.close()
    return rows

db_path = "/tmp/week8_demo_conversations.db"
if os.path.exists(db_path):
    os.remove(db_path)  # start clean so this cell is reproducible on rerun

save_history_sqlite(history[:4], db_path=db_path)
roundtrip = load_history_sqlite(db_path=db_path)
print(roundtrip)
print("roundtrip matches original:", roundtrip == history[:4])
os.remove(db_path)


[('user', 'question 1 about the order'), ('assistant', 'answer 1'), ('user', 'question 2 about the order'), ('assistant', 'answer 2')]
roundtrip matches original: True


In [ ]:
import re

_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_PHONE_RE = re.compile(r"(?<!\w)\+?\d[\d\-\s]{7,}\d(?!\w)")

def redact_pii(text: str) -> str:
    text = _EMAIL_RE.sub("[redacted-email]", text)
    text = _PHONE_RE.sub("[redacted-phone]", text)
    return text

samples = [
    "my email is a.kim@example.com, call me at 555-123-4567 too",
    "reach me at +1 415 555 0199 or backup jane.doe+work@corp.co.kr",
    # spelled-out digits: the regex only looks for digit characters, so
    # this sails through completely untouched -- a real, verified gap.
    "my number is five five five, one two three, four five six seven",
]
for s in samples:
    print(redact_pii(s))


my email is [redacted-email], call me at [redacted-phone] too
reach me at [redacted-phone] or backup [redacted-email]
my number is five five five, one two three, four five six seven
